# Differential Equations — Session 29
## Section 6.4: Special Functions

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Identify Bessel's and Legendre's equations; explain why Bessel requires Frobenius; derive its indicial roots; interpret $J_\nu$, $Y_\nu$, $I_\nu$, and $K_\nu$; derive the Legendre recurrence; explain termination for integer degree; and visualize zeros and orthogonality.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–15 min | Why special functions arise |
| 15–40 min | Bessel equation and Frobenius |
| 40–58 min | Zeros and modified functions |
| 58–78 min | Legendre equation |
| 78–88 min | Orthogonality |
| 88–90 min | Exit check |

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.special import jv, yv, iv, kv, eval_legendre, jn_zeros
from IPython.display import display, Markdown
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=6, suppress=True)
def polynomial_value(coefficients, x):
    x = np.asarray(x, dtype=float)
    total = np.zeros_like(x)
    for n, c in enumerate(coefficients):
        total += c*x**n
    return total
print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 6.4-A — Bessel equation

$$
x^2y''+xy'+(x^2-\nu^2)y=0.
$$

Zero is regular singular, with indicial roots $r=\pm\nu$.

### Definition 6.4-B — Bessel functions

$$
J_\nu(x)
=
\sum_{m=0}^{\infty}
\frac{(-1)^m}{m!\Gamma(m+\nu+1)}
\left(\frac{x}{2}\right)^{2m+\nu}.
$$

A second independent solution is denoted $Y_\nu$. On $x>0$,
$$
y=c_1J_\nu(x)+c_2Y_\nu(x).
$$

### Definition 6.4-C — Modified Bessel equation

$$
x^2y''+xy'-(x^2+\nu^2)y=0,
$$
with independent solutions $I_\nu$ and $K_\nu$.

### Definition 6.4-D — Legendre equation

$$
(1-x^2)y''-2xy'+n(n+1)y=0.
$$

For nonnegative integer $n$, one series terminates and gives $P_n(x)$.

### Theorem 6.4-E — Rodrigues' formula

$$
P_n(x)=
\frac{1}{2^nn!}\frac{d^n}{dx^n}(x^2-1)^n.
$$

### Theorem 6.4-F — Orthogonality

For $m\ne n$,
$$
\int_{-1}^{1}P_m(x)P_n(x)\,dx=0.
$$

### Classroom Checkpoint — Identify the Bessel Order

What is the order of

$$
x^2y''+xy'+(x^2-16)y=0?
$$

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Bessel functions from Frobenius

For the root $r=\nu$, the recurrence is
$$
c_{k+2}=
-\frac{c_k}{(k+2)(k+2+2\nu)}.
$$

In [ ]:
def bessel_series(nu=0.0, terms=12, x=None):
    if x is None:
        x = np.linspace(0.001, 20, 800)
    total = np.zeros_like(np.asarray(x, dtype=float))
    for m in range(terms):
        total += (-1)**m*(x/2)**(2*m+nu)/(math.factorial(m)*math.gamma(m+nu+1))
    return total

x = np.linspace(0.001, 20, 800)
for nu in [0, 1, 2]:
    plt.plot(x, jv(nu, x), label=fr"$J_{nu}$")
plt.axhline(0, linestyle="--")
plt.legend()
plt.title("Bessel functions of the first kind")
plt.show()

In [ ]:
def bessel_series_explorer(nu=0.0, terms=8):
    x = np.linspace(0.001, 15, 700)
    plt.plot(x, jv(nu, x), label="SciPy")
    plt.plot(x, bessel_series(nu, terms, x), linestyle="--", label=f"{terms} terms")
    plt.ylim(-2, 2)
    plt.legend()
    plt.title(fr"$J_{{{nu}}}(x)$")
    plt.show()
if WIDGETS_AVAILABLE:
    interact(bessel_series_explorer,
             nu=FloatSlider(min=0, max=4, step=0.5, value=0),
             terms=IntSlider(min=2, max=35, value=8))
else:
    bessel_series_explorer()

## 2. First and second kinds

$J_\nu$ is finite at zero for $\nu\ge0$, while $Y_\nu$ is generally singular there.

In [ ]:
x = np.linspace(0.05, 15, 700)
plt.plot(x, jv(0, x), label=r"$J_0$")
plt.plot(x, yv(0, x), label=r"$Y_0$")
plt.ylim(-3, 2)
plt.axhline(0, linestyle="--")
plt.legend()
plt.show()

## 3. Zeros as radial eigenvalues

In [ ]:
z0 = jn_zeros(0, 5)
z1 = jn_zeros(1, 5)
print("J0 zeros:", z0)
print("J1 zeros:", z1)
x = np.linspace(0, 18, 800)
plt.plot(x, jv(0, x), label=r"$J_0$")
plt.scatter(z0, np.zeros_like(z0), s=70, label="zeros")
plt.axhline(0, linestyle="--")
plt.legend()
plt.show()

## 4. Modified Bessel functions

$I_\nu$ grows and $K_\nu$ decays for large positive $x$.

In [ ]:
x = np.linspace(0.1, 8, 600)
plt.semilogy(x, iv(0, x), label=r"$I_0$")
plt.semilogy(x, kv(0, x), label=r"$K_0$")
plt.legend()
plt.title("Modified Bessel functions")
plt.show()

## Optional extension — Half-integer order

$$
J_{1/2}(x)=\sqrt{\frac{2}{\pi x}}\sin x.
$$

In [ ]:
x = np.linspace(0.05, 15, 700)
formula = np.sqrt(2/(np.pi*x))*np.sin(x)
plt.plot(x, jv(0.5, x), label=r"$J_{1/2}$")
plt.plot(x, formula, linestyle="--", label="elementary formula")
plt.legend(); plt.show()
print("Maximum difference:", np.max(np.abs(jv(0.5, x)-formula)))

## 5. Legendre recurrence and termination

The coefficient recurrence is
$$
c_{j+2}
=
-\frac{(n-j)(n+j+1)}{(j+2)(j+1)}c_j.
$$

For integer $n$, the matching parity chain terminates.

In [ ]:
x = np.linspace(-1, 1, 700)
for n in range(6):
    plt.plot(x, eval_legendre(n, x), label=fr"$P_{n}$")
plt.axhline(0, linestyle="--")
plt.legend(ncol=2)
plt.title("First six Legendre polynomials")
plt.show()

In [ ]:
def legendre_explorer(n=3):
    x = np.linspace(-1, 1, 700)
    plt.plot(x, eval_legendre(n, x), linewidth=2)
    plt.axhline(0, linestyle="--")
    plt.title(fr"$P_{n}(x)$")
    plt.show()
    print("P_n(1) =", eval_legendre(n, 1))
    print("P_n(-1) =", eval_legendre(n, -1))
if WIDGETS_AVAILABLE:
    interact(legendre_explorer, n=IntSlider(min=0, max=15, value=3))
else:
    legendre_explorer()

## 6. Orthogonality matrix

$$
\int_{-1}^{1}P_mP_n\,dx
=
0\quad(m\ne n),
\qquad
\int_{-1}^{1}P_n^2\,dx
=
\frac{2}{2n+1}.
$$

In [ ]:
G = np.zeros((6, 6))
for m in range(6):
    for n in range(6):
        G[m, n] = quad(lambda x: eval_legendre(m, x)*eval_legendre(n, x), -1, 1)[0]
print(G)
plt.imshow(np.abs(G))
plt.colorbar(label="absolute inner product")
plt.title("Legendre orthogonality matrix")
plt.show()

## 7. Rodrigues' formula

In [ ]:
x = sp.symbols("x")
for n in range(6):
    Pn = sp.expand(sp.diff((x**2-1)**n, x, n)/(2**n*sp.factorial(n)))
    print(f"P_{n}(x) =")
    display(Pn)

## Exit check

$$
x^2y''+xy'+(x^2-9)y=0
$$

is Bessel's equation of order $3$, so
$$
y=c_1J_3(x)+c_2Y_3(x).
$$

## Classroom Checkpoint — Closing Reflection

Before continuing, try to state the central method or theorem of this lesson, including its assumptions and one situation in which it is useful.

> Discuss first; run the next cell for an instructor summary.